# score

> Run every rule, weigh the findings, and report worst first

In [ ]:
#| default_exp score

In [ ]:
#| hide
from nbdev.showdoc import *

`score_text` runs the registered rules on Markdown and returns findings in descending weight order. `score_path` reads a Markdown file or a notebook's Markdown cells.

Density is the total finding weight per 100 scored words. `worst` is the highest individual weight. Both help identify prose to review, but neither establishes whether the explanation is clear or complete.

In [ ]:
#| export
from fastcore.utils import *
from fastcore.tools import lnhash_at
from fastcore.nbio import read_nb
from slopometer.core import *
from slopometer.segment import *
from slopometer.lexicon import *
from slopometer.syntax import *
from slopometer.para import *

In [ ]:
from fastcore.test import *
from nbdev.config import get_config
import tempfile

## Over-structuring

`find_overstructure` flags documents with at least two headings and fewer than 40 paragraph words per heading. This approximates tell 25's warning against imposing headings on short documents.

In [ ]:
#| export
@rule('overstructure', tell=25, weight=SMELL, level='doc')
def find_overstructure(blocks, docs):
    "More headings than the prose beneath them justifies"
    nh,nw = n_headings(blocks), prose_words(blocks)
    if nh < 2 or nw/nh >= 40: return []
    hs = [b for b in blocks if b.kind == 'heading']
    return [Finding('overstructure', 25, hs[0].start, hs[0].start+len(hs[0].txt),
        f'{nh} headings on {nw} words of prose', SMELL)]

## The runner

`run_rules` dispatches each rule according to its level:

- Phrase rules read scrubbed text from every block, including headings and list items.
- Sentence and paragraph rules read parsed prose blocks.
- Document rules read all blocks in one call.

The runner converts local finding offsets to document offsets. It adds the block's start, plus the sentence's offset for sentence rules. Segmentation and scrubbing preserve these positions.

In [ ]:
#| export
def _rebase(fs, off):
    for f in fs:
        f.start += off
        f.end += off
    return fs

def run_rules(txt):
    "Every registered rule's findings for markdown `txt`, rebased to document offsets"
    blocks = segment(txt)
    docs = parse_blocks(blocks)
    out = []
    for b,d in zip(blocks, docs):
        sc = scrub(b.txt)
        for r in rules.values():
            if r.level == 'phrase': out += _rebase(r(sc), b.start)
            elif d is None: continue
            elif r.level == 'sentence':
                for s in d.sents: out += _rebase(r(s), b.start+s.start_char)
            elif r.level == 'para': out += _rebase(r(d), b.start)
    for r in rules.values():
        if r.level == 'doc': out += r(blocks, docs)
    return sorted(out, key=lambda f: (-f.weight, f.start))


## The result

`Result` displays a score header followed by findings, worst first. File findings include `lineno|hash|` addresses for exhash editing tools.

The density denominator counts words in paragraphs, headings, and list items. It excludes Markdown markers, code blocks, and other content discarded by segmentation. The over-structuring rule has a separate paragraph-only count.

In [ ]:
#| export
class Result:
    def __init__(self,
        txt, # The scored document
        findings, # `Finding`s, worst first
        words, # Word count across all scored blocks
        path=None, # Source file, when scoring a file
        min_words=150, # Minimum scored words; 0 disables the cutoff
        cells=None, # Notebook (document offset, cell) pairs; None for plain text
    ):
        store_attr()
        for f in findings:
            if len(f.text) == f.end-f.start and '\n' not in txt[f.start:f.end]: f.text = txt[f.start:f.end]
    @property
    def too_short(self): return self.words < self.min_words
    @property
    def total(self): return None if self.too_short else sum(f.weight for f in self.findings)
    @property
    def density(self): return None if self.too_short else round(100*self.total/max(self.words, 1), 1)
    @property
    def worst(self): return None if self.too_short else max((f.weight for f in self.findings), default=0)
    def location(self, f):
        "Source line and exhash address; notebooks also include cell ID and zero-based index"
        txt, off, cell = self.txt, 0, None
        for start,c in reversed(self.cells or []):
            if start <= f.start < start + len(c.source):
                txt, off, cell = c.source, start, c
                break
        ln = txt[:f.start-off].count('\n')+1
        addr = lnhash_at(txt, ln) if self.path else f'{ln}:'
        if cell is None: return dict(line=ln, address=addr)
        return dict(cell_id=cell.id, cell_index=cell.idx_, line=ln, address=f'{cell.id}:{addr}')
    def _addr(self, f): return self.location(f)['address']
    def __repr__(self):
        if self.too_short: hdr = f'too short to meter ({self.words} scored words; minimum {self.min_words})'
        else: hdr = f'density {self.density} (weight {self.total} on {self.words} prose words), worst {self.worst}'
        if self.path: hdr = f'{self.path}: {hdr}'
        return '\n'.join([hdr] + [f'{self._addr(f)} {f!r}' for f in self.findings])

def score_text(txt, min_words=150):
    "Score markdown `txt`, skipping rule evaluation below `min_words` scored words"
    if min_words < 0: raise ValueError('min_words must be non-negative')
    words = scored_words(segment(txt))
    return Result(txt, run_rules(txt) if words >= min_words else [], words, min_words=min_words)

def score_path(p, min_words=150):
    "Score Markdown or an `.ipynb` file's Markdown cells; report exhash addresses"
    p = Path(p).expanduser()
    cells = None
    if p.suffix.lower() == '.ipynb':
        cells, off = [], 0
        for c in read_nb(p).cells:
            if c.cell_type != 'markdown': continue
            cells.append((off, c))
            off += len(c.source) + 2
        txt = '\n\n'.join(c.source for _,c in cells)
    else: txt = p.read_text()
    res = score_text(txt, min_words=min_words)
    res.path, res.cells = p, cells
    return res


The default minimum is 150 scored words. Shorter inputs return their word count with no score or findings. Code does not count toward this minimum. Set `min_words=0` to score short examples.


In [ ]:
short = score_text('robust widgets')
test_eq(short.words, 2)
assert short.too_short
test_eq((short.total, short.density, short.worst), (None, None, None))
test_eq(short.findings, [])
test_eq(str(short), 'too short to meter (2 scored words; minimum 150)')
assert score_text('').too_short
assert score_text('word ' * 149).too_short
assert not score_text('word ' * 150).too_short
assert score_text('robust widgets\n\n```python\n' + 'code ' * 200 + '\n```').too_short
assert score_text('robust widgets', min_words=3).too_short
assert not score_text('robust widgets', min_words=2).too_short
test_eq(score_text('robust widgets', min_words=0).total, 10)
with expect_fail(ValueError, contains='min_words'):
    score_text('robust widgets', min_words=-1)
short


too short to meter (2 scored words; minimum 150)

Markdown markers do not change the scored-word count:

In [ ]:
for prefix in ('', '- ', '* ', '+ ', '1. ', '2) ', '# ', '## '):
    r = score_text(prefix + 'robust widgets', min_words=0)
    test_eq(r.words, 2)
    test_eq(r.total, 10)
    test_eq(r.density, 500.0)


Adding clean prose lowers density without changing the finding weight:

In [ ]:
mixed = '# robust widgets\n\nRead the guide.\n\n- fast startup\n- clean shutdown\n'
r = score_text(mixed, min_words=0)
test_eq(r.words, 9)
r_more = score_text(mixed + '- quiet exits\n', min_words=0)
test_eq(r_more.words, 11)
test_eq(r_more.total, r.total)
assert r_more.density < r.density

Excluded code cannot dilute the score, and files use the same denominator as text.

In [ ]:
with_code = mixed + '\n```python\nrobust paradigm\n```\n\n    more excluded code\n'
r_code = score_text(with_code, min_words=0)
test_eq((r_code.words, r_code.density), (r.words, r.density))
with tempfile.TemporaryDirectory() as d:
    p = Path(d)/'README.md'
    p.write_text(with_code)
    skipped = score_path(p)
    assert skipped.too_short
    test_eq(str(skipped), f'{p}: too short to meter (9 scored words; minimum 150)')
    rp = score_path(p, min_words=0)
    test_eq((rp.words, rp.density), (r.words, r.density))
test_eq(score_text('', min_words=0).density, 0.0)

`score_path` joins a notebook's Markdown cells with blank lines and scores them as one document. It excludes code, outputs, and raw cells. The word minimum applies to the combined prose.

Findings use offsets into that combined text. `Result.location` supplies the cell ID, zero-based cell index, one-based local line, and exhash address.

In [ ]:
from fastcore.nbio import new_nb, mk_cell, write_nb


In [ ]:
nb = new_nb([
    mk_cell('# Widgets', 'markdown', id='heading'),
    mk_cell('paradigm ' * 200, outputs=[dict(output_type='stream', name='stdout', text='seamless ' * 200)]),
    mk_cell('', 'markdown', id='empty'),
    mk_cell(['Read the guide.\n', 'Robust widgets.'], 'markdown', id='body'),
    mk_cell('comprehensive ' * 200, 'raw')])
with tempfile.TemporaryDirectory() as d:
    p = Path(d)/'guide.ipynb'
    write_nb(nb, p)
    nr = score_path(p, min_words=0)
    test_eq(nr.txt, '# Widgets\n\n\n\nRead the guide.\nRobust widgets.')
    test_eq(nr.words, 6)
    test_eq([f.text for f in nr.findings], ['Robust'])
    loc = nr.location(nr.findings[0])
    test_eq((loc['cell_id'], loc['cell_index'], loc['line']), ('body', 3, 2))
    test_eq(loc['address'], 'body:' + lnhash_at(nb.cells[3].source, 2))
    assert loc['address'] in str(nr)
    test_eq(nr.txt[nr.findings[0].start:nr.findings[0].end], 'Robust')
    assert score_path(p).too_short
    assert not score_path(p, min_words=6).too_short
    test_eq(nr.density, score_text(nr.txt, min_words=0).density)
    write_nb(new_nb([nb.cells[1], nb.cells[4]]), p)
    empty = score_path(p, min_words=0)
    test_eq((empty.txt, empty.words, empty.density, empty.findings), ('', 0, 0.0, []))
loc


{'cell_id': 'body', 'cell_index': 3, 'line': 2, 'address': 'body:2|f030|'}

`score_many` uses threads. spaCy's C operations release the GIL during scoring. In the original benchmark, 8 workers scored a batch 2.7 times faster than serial execution. Sixteen workers were slower, and process pools used more memory without improving speed. The default is 8 workers.

In [ ]:
#| export
def score_many(txts, n_workers=8, min_words=150):
    "Score texts in parallel threads, applying `min_words` to each text"
    return parallel(score_text, txts, n_workers=n_workers, threadpool=True, min_words=min_words)

In [ ]:
pair = ['The gateway never kills an unresponsive kernel.', "It isn't just fast - it's seamless."]
assert all(r.too_short for r in score_many(pair))
test_eq([r.density for r in score_many(pair, min_words=0)], [score_text(t, min_words=0).density for t in pair])
test_eq([r.too_short for r in score_many(pair, min_words=8)], [score_text(t, min_words=8).too_short for t in pair])


In [ ]:
example = score_text('''# widget

A tool for widgets. Call `enhance()` to start.

## Features

widget leverages a robust paradigm to deliver results.
It parses the config; it writes the output.

- fast startup
- fast shutdown
- fast restarts
''', min_words=0)
test_eq([f.weight for f in example.findings if f.rule == 'semi_splice'], [4])
example

density 156.2 (weight 50 on 32 prose words), worst 10
7: [10] banned: 'leverages' -> 'use'
7: [10] banned: 'robust' -> 'strong'
7: [10] banned: 'paradigm'
7: [10] banned: 'deliver results'
8: [4] semi_splice (tell 1, splices): ';'
1: [3] overstructure (tell 25, over-structuring): '2 headings on 24 words of prose'
10: [3] bullet_mold (tell 20, forced symmetry): '- fast startup\n- fast shutdown\n- fast restarts'

## The drift test

Each numbered tell in `write_docs` must have a registered rule or an entry in the unscoreable registry. This check fails when a new tell has neither. It runs when `aai_coding` is installed and reports when it skips.

In [ ]:
try:
    import aai_coding.write_docs as _wd
    doc_tells = {int(n) for n in re.findall(r'^(\d+)\.', _wd.__doc__, flags=re.M)}
    covered = {r.tell for r in rules.values() if r.tell is not None} | set(unscoreable)
    test_eq(doc_tells - covered, set())
    print(f'{len(doc_tells)} tells in write_docs, every one mapped to a rule or the registry')
except ImportError: print('aai_coding not installed: drift test skipped, it runs on team machines')

26 tells in write_docs, every one mapped to a rule or the registry


## Calibration

`write_docs` shows the same design-doc section before and after editing. The meter uses this pair for calibration. The clean passage should score near zero. The original should score high and include the intended kill-tier findings. The fixtures omit the skill's annotation numbers.

In [ ]:
AFTER = "`GatewayKernel` ties the three lower layers together. The ready-wait runs once per kernel, in `start`. `watch` polls the process and the heartbeat. A process that dies unexpectedly broadcasts the synthesized `dead` status. Three missed heartbeats mark the kernel `unresponsive` in its model, with the next echo clearing the mark.\n\nThe gateway never kills an unresponsive kernel. A kernel becomes `dead` only when its process exits. `restart` terminates and respawns with fresh ports in a new process. Clients see `restarting`, then `starting` once the new kernel is ready."

BEFORE = "This section describes how `GatewayKernel` manages the kernel lifecycle. `GatewayKernel` ties the three lower layers together, and its `start` is the only place the ready-wait runs: once per kernel, ever. The core mechanism: `watch`. It isn't just a poller - it's the liveness authority. Furthermore, it polls the process and the heartbeat: a process that dies unexpectedly broadcasts the synthesized `dead` status, and three missed beats mark the kernel `unresponsive` in its model. The distinction is worth being precise about. That marking is observational only - only process exit means dead - and it clears itself on the next echo. So what does `restart` actually do? It terminates and respawns; the kernel gains fresh ports, fresh channels, and a fresh interpreter via the new process, so the channel set is rebuilt and clients simply see `restarting` then a fresh welcome-backed ready kernel."

ra, rb = score_text(AFTER, min_words=0), score_text(BEFORE, min_words=0)
ra

density 0.0 (weight 0 on 87 prose words), worst 0

In [ ]:
rb

density 45.1 (weight 64 on 142 prose words), worst 12
1: [12] sent_len (tell 1, splices): 'It terminates and respawns; the kernel gains fresh ports, fresh channels, and a fresh interpreter via the new process, so the channel set is rebuilt and clients simply see `restarting` then a fresh welcome-backed ready kernel.'
1: [10] notxbuty (tell 16, not-X-but-Y): "isn't just a"
1: [4] sent_len (tell 1, splices): 'Furthermore, it polls the process and the heartbeat: a process that dies unexpectedly broadcasts the synthesized `dead` status, and three missed beats mark the kernel `unresponsive` in its model.'
1: [4] semi_splice (tell 1, splices): ';'
1: [3] throat_clearing (tell 13, throat-clearing): 'This section describes'
1: [3] announce (tell 15, announce-then-deliver): 'The core mechanism: '
1: [3] transitions (tell 19, filler transitions): 'Furthermore' -> 'and'
1: [3] variation (tell 4, elegant variation): 'heartbeat ... beats'
1: [3] appraisal (tell 21, appraisal preamble): 'worth being

Check these properties without fixing the exact score for the original passage:

In [ ]:
test_eq(ra.total, 0)
assert rb.density > 30 and rb.worst >= 10
hit = {f.tell for f in rb.findings if f.tell is not None}
assert {1, 13, 16, 18, 19, 21} <= hit
sorted(hit)

[1, 3, 4, 6, 13, 15, 16, 18, 19, 20, 21, 23]

## The register, measured

`samples/` contains four drafts from warmpy's design:

- `theory1.md` uses aphorisms and elaborate phrasing.
- `theory2.md` mechanically splits the same explanation into short sentences.
- `theory3.md` rewrites it as a coherent narrative.
- `theory4.md` adapts that narrative to reference prose.

`theory4` names code symbols and makes references clear, as in "`yourcommand` becomes a small program that starts fast". It scores low without reducing every sentence to a fragment.

`theory2` scores lower than `theory3` despite being harder to understand. "The server is a cache, and doubt means disposal" passes the surface rules without explaining what to do. Missing context has no span for the meter to flag. A low score is not a verdict on the explanation.


In [ ]:
sd = get_config().config_path/'samples'
t1,t2,t3,t4 = (score_path(sd/f'theory{i}.md', min_words=0) for i in (1, 2, 3, 4))
[(f'theory{i}', r.density, r.worst) for i,r in zip((1, 2, 3, 4), (t1, t2, t3, t4))]


[('theory1', 47.3, 53),
 ('theory2', 4.5, 3),
 ('theory3', 11.1, 12),
 ('theory4', 1.3, 4)]

## Referential load, a prototype

Splitting sentences can leave references unclear. "It reads the same stdin" needs an antecedent for "it". "The record of completed imports dies with the process" assumes the reader knows which record. Entity-based coherence models, such as [Barzilay and Lapata (2008)](https://aclanthology.org/J08-1001/), study how documents introduce entities and refer to them later.

`refload` measures two possible indicators:

- The pronoun rate (`pron`) is the fraction of sentences whose main subject is a pronoun.
- The cold-definite rate (`cold`) is the fraction of "the X" phrases where X has not appeared earlier in the document. This count excludes "the same X" and placeholder tokens used for inline code.

These are experimental measurements, not scoring rules.

In [ ]:
def refload(txt):
    n_sent = n_pron = n_def = n_cold = 0
    seen = set()
    for d in (x for x in parse_blocks(segment(txt)) if x is not None):
        for s in d.sents:
            n_sent += 1
            if any(t.dep_ in ('nsubj', 'nsubjpass') and t.pos_ == 'PRON' and t.head.dep_ == 'ROOT' for t in s): n_pron += 1
        for t in d:
            if t.pos_ not in ('NOUN', 'PROPN') or len(set(t.lower_)) == 1: continue
            lem = t.lemma_.lower()
            kids = [c.lower_ for c in t.children]
            if 'the' in kids and 'same' not in kids:
                n_def += 1
                if lem not in seen: n_cold += 1
            seen.add(lem)
    return dict(sents=n_sent, pron=round(n_pron/max(n_sent, 1), 2), defs=n_def, cold=round(n_cold/max(n_def, 1), 2))

In [ ]:
texts = {f'theory{i}': (sd/f'theory{i}.md').read_text() for i in (1, 2, 3, 4)} | dict(AFTER=AFTER, BEFORE=BEFORE)
{k: refload(t) for k,t in texts.items()}

{'theory1': {'sents': 23, 'pron': 0.09, 'defs': 34, 'cold': 0.56},
 'theory2': {'sents': 33, 'pron': 0.06, 'defs': 35, 'cold': 0.46},
 'theory3': {'sents': 35, 'pron': 0.03, 'defs': 40, 'cold': 0.28},
 'theory4': {'sents': 33, 'pron': 0.03, 'defs': 23, 'cold': 0.3},
 'AFTER': {'sents': 9, 'pron': 0.0, 'defs': 9, 'cold': 0.89},
 'BEFORE': {'sents': 9, 'pron': 0.11, 'defs': 14, 'cold': 0.79}}

The cold-definite rate is higher in `theory2` than in `theory3` or `theory4`. The pronoun rate differs little across these rewrites. Neither measurement reliably establishes whether references are clear.

The clean calibration excerpt has a cold-definite rate of 0.89. Its surrounding document introduced the entities, but the meter sees only the excerpt. Evaluating this measure needs whole documents and human judgments of their clarity. It remains a measurement rather than a rule.

In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()